In [ ]:
from pathlib import Path
import sys
import json
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

from autoencoder_anomaly_detector import (
    binary_metrics,
    classify,
    load_ae_run,
    list_ae_runs,
    load_window_run,
    mse_reconstruction_scores,
    plot_thresholds,
    roc_auc_binary,
)

try:
    from sklearn.ensemble import IsolationForest
except ImportError as exc:
    raise ImportError('scikit-learn is required for Isolation Forest runs. Install with: pip install scikit-learn') from exc

def resolve_path(path_value: str | Path) -> Path:
    path_obj = Path(path_value)
    if path_obj.is_absolute():
        return path_obj
    return (PROJECT_ROOT / path_obj).resolve()

def to_relative_path(path_value: str | Path) -> str:
    path_obj = Path(path_value)
    if not path_obj.is_absolute():
        return path_obj.as_posix()
    try:
        return path_obj.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
    except Exception:
        return str(path_value)

In [ ]:
# Choose model type and run index
MODEL_TYPE = 'isolation_forest'  # 'autoencoder' or 'isolation_forest'
MODEL_INDEX = 0

def list_isolation_forest_runs(model_root: Path):
    runs = []
    if not model_root.exists():
        return runs
    for run_dir in sorted(model_root.iterdir(), reverse=True):
        if not run_dir.is_dir():
            continue
        summary_path = run_dir / 'summary.json'
        summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
        runs.append({
            'run_name': run_dir.name,
            'run_path': run_dir,
            'summary': summary,
        })
    return runs

if MODEL_TYPE == 'autoencoder':
    MODEL_ROOT = PROJECT_ROOT / 'models' / 'autoencoder'
    available_runs = list_ae_runs(MODEL_ROOT)
elif MODEL_TYPE == 'isolation_forest':
    MODEL_ROOT = PROJECT_ROOT / 'models' / 'isolation_forest'
    available_runs = list_isolation_forest_runs(MODEL_ROOT)
else:
    raise ValueError("MODEL_TYPE must be 'autoencoder' or 'isolation_forest'.")

if not available_runs:
    raise ValueError(f'No trained {MODEL_TYPE} models found in {MODEL_ROOT}.')

print(f'Found {len(available_runs)} trained {MODEL_TYPE} model(s):')
print('\nAvailable models (sorted by most recent):')
for i, run_info in enumerate(available_runs):
    run_name = run_info['run_name']
    summary = run_info['summary']
    train_samples = summary.get('train_samples', 'N/A')
    threshold = summary.get('threshold', 'N/A')
    print(f'  [{i}] {run_name} | train_samples={train_samples}, threshold={threshold}')

In [ ]:
# Select which model to test
selected_run_info = available_runs[MODEL_INDEX]
selected_run_name = selected_run_info['run_name']
selected_run_path = selected_run_info['run_path']
MODEL_PLOTS_DIR = selected_run_path / 'plots'
MODEL_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Loading model: {selected_run_name}')
summary = selected_run_info['summary']
window_run_value = summary.get('source_data_run')
window_run_dir = resolve_path(window_run_value) if window_run_value else None

if MODEL_TYPE == 'autoencoder':
    ae_run = load_ae_run(selected_run_path)
    encoder = ae_run['encoder']
    decoder = ae_run['decoder']
    train_scores_saved = ae_run['train_scores']
elif MODEL_TYPE == 'isolation_forest':
    model_path = selected_run_path / 'isolation_forest.pkl'
    if not model_path.exists():
        raise FileNotFoundError(f'Missing model file at {model_path}')
    with model_path.open('rb') as f:
        iso = pickle.load(f)
    train_scores_saved = np.load(selected_run_path / 'train_scores.npy') if (selected_run_path / 'train_scores.npy').exists() else None
else:
    raise ValueError("MODEL_TYPE must be 'autoencoder' or 'isolation_forest'.")

if window_run_dir is None:
    raise ValueError('source_data_run missing from summary.json for the selected run.')

print(f'Model loaded from: {selected_run_path}')
print(f'Processed window run: {window_run_dir}')
print(f'Plots directory: {MODEL_PLOTS_DIR}')

In [ ]:
train_windows, val_windows, test_windows, train_labels, val_labels, test_labels, metadata = load_window_run(window_run_dir)

feature_count = len(metadata.get('feature_cols', []))
print(f'Loaded window arrays: train={train_windows.shape}, val={val_windows.shape}, test={test_windows.shape}')
print(f'Feature count from processed windows: {feature_count}')
print(f'Test labels - normal: {int((test_labels == 0).sum())}, anomaly: {int((test_labels == 1).sum())}')

In [ ]:
# Get batch size from summary (AE only)
train_cfg_dict = summary.get('train_config', {})
batch_size = train_cfg_dict.get('batch_size', 1024)

if MODEL_TYPE == 'autoencoder':
    # Compute test reconstruction scores
    test_scores = mse_reconstruction_scores(encoder, decoder, test_windows, batch_size=batch_size)
elif MODEL_TYPE == 'isolation_forest':
    test_flat = test_windows.reshape(len(test_windows), -1)
    # Isolation Forest score_samples: higher = more normal. Flip to make higher = more anomalous.
    test_scores = -iso.score_samples(test_flat)
else:
    raise ValueError("MODEL_TYPE must be 'autoencoder' or 'isolation_forest'.")

# Load threshold from summary
threshold = summary.get('threshold')
if threshold is None:
    raise ValueError('No threshold found in model summary. Check the model run.')

print(f'Threshold: {threshold:.6f}')
print(f'Test scores - min: {test_scores.min():.6f}, max: {test_scores.max():.6f}, mean: {test_scores.mean():.6f}')

In [ ]:
# Classify test data
test_preds = classify(test_scores, threshold)

print(f'Test predictions - normal: {int((test_preds == 0).sum())}, anomaly: {int((test_preds == 1).sum())}')

In [ ]:
metrics = binary_metrics(test_labels, test_preds)
metrics['roc_auc'] = roc_auc_binary(test_labels, test_scores)

print('Test Metrics:')
display(pd.DataFrame([metrics]))

# Show training metrics from summary if available
train_metrics_saved = summary.get('train_metrics', {})
if train_metrics_saved:
    print('\nTraining Metrics (from training run):')
    display(pd.DataFrame([train_metrics_saved]))

In [ ]:
if MODEL_TYPE == 'autoencoder':
    fig, ax = plot_thresholds(
        scores=test_scores,
        labels=test_labels,
        threshold=threshold,
        title=f'Reconstruction Error Distribution - Model: {selected_run_name}',
    )
    threshold_plot_path = MODEL_PLOTS_DIR / 'reconstruction_error_distribution.png'
else:
    normal_scores = test_scores[test_labels == 0]
    anomaly_scores = test_scores[test_labels == 1]
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.hist(normal_scores, bins=50, alpha=0.6, label=f'Normal (n={len(normal_scores)})', color='blue', edgecolor='black')
    if len(anomaly_scores) > 0:
        ax.hist(anomaly_scores, bins=50, alpha=0.6, label=f'Anomaly (n={len(anomaly_scores)})', color='red', edgecolor='black')
    ax.axvline(threshold, color='green', linestyle='--', linewidth=2, label=f'Threshold = {threshold:.6f}')
    ax.set_xlabel('Isolation Forest anomaly score')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Isolation Forest Score Distribution - Model: {selected_run_name}')
    ax.legend()
    ax.grid(alpha=0.3)
    threshold_plot_path = MODEL_PLOTS_DIR / 'isolation_forest_score_distribution.png'

fig.savefig(threshold_plot_path, dpi=160, bbox_inches='tight')
print(f'Saved plot: {threshold_plot_path}')
plt.show()

In [ ]:
# Summary information and model metrics comparison
print('=' * 60)
print('MODEL SUMMARY AND METRICS COMPARISON')
print('=' * 60)

print('\nModel Configuration:')
vae_cfg = summary.get('vae_config', {})
display(pd.json_normalize(vae_cfg, sep='.').T.rename(columns={0: 'value'}))

print('\nThreshold Configuration:')
threshold_cfg = summary.get('threshold_config', {})
display(pd.json_normalize(threshold_cfg, sep='.').T.rename(columns={0: 'value'}))

# Show performance comparison
print('\nPerformance Summary:')
train_metrics_summary = summary.get('train_metrics', {})
print('\nTraining Metrics (from training run):')
if train_metrics_summary:
    train_df = pd.DataFrame([train_metrics_summary])
    display(train_df)
else:
    print('  No training metrics saved')

print('\nTest Metrics (current evaluation):')
display(pd.DataFrame([metrics]))

In [ ]:
# Real-case view over processed windows: actual anomaly labels vs model predictions, score dynamics.
plot_df = pd.DataFrame({
    'window_index': np.arange(len(test_scores)),
    'test_score': test_scores,
    'actual': test_labels,
    'predicted': test_preds,
})
plot_df['false_negative'] = ((plot_df['actual'] == 1) & (plot_df['predicted'] == 0)).astype(int)

# Rolling false-negative density to indicate how concentrated misses are across the windowed test run.
window = max(10, int(len(plot_df) * 0.01))
plot_df['fn_density'] = plot_df['false_negative'].rolling(window=window, min_periods=1).mean()

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True, gridspec_kw={'height_ratios': [1.1, 1.4]})

# Filled step plots for actual and predicted events
axes[0].fill_between(plot_df['window_index'], 0, plot_df['predicted'], step='post', label='predicted anomaly', alpha=0.5, color='blue')
axes[0].set_ylim(-0.1, 1.2)
axes[0].set_ylabel('event')
axes[0].set_title('Actual vs Predicted Anomaly Events Over Windows')
axes[0].legend(loc='upper right')
axes[0].grid(alpha=0.25)

score_label = 'reconstruction score' if MODEL_TYPE == 'autoencoder' else 'anomaly score'
timeline_title = 'Score Timeline Across Processed Windows' if MODEL_TYPE == 'autoencoder' else 'Isolation Forest Score Timeline Across Processed Windows'
axes[1].plot(plot_df['window_index'], plot_df['test_score'], label=score_label, linewidth=1.2)
axes[1].axhline(threshold, linestyle='--', linewidth=1.5, label=f'threshold={threshold:.4f}')
for _, row in plot_df[plot_df['actual'] == 1].iterrows():
    axes[1].axvspan(row['window_index'], row['window_index'], color='orange', alpha=0.08)
axes[1].set_ylabel('score')
axes[1].set_title(timeline_title)
axes[1].legend(loc='upper right')
axes[1].grid(alpha=0.25)

timeline_plot_path = (
    MODEL_PLOTS_DIR / 'anomaly_timeline_with_fn_density.png'
    if MODEL_TYPE == 'autoencoder'
    else MODEL_PLOTS_DIR / 'isolation_forest_timeline.png'
 )
fig.savefig(timeline_plot_path, dpi=160, bbox_inches='tight')
print(f'Saved plot: {timeline_plot_path}')

plt.tight_layout()
plt.show()


In [ ]:
# Save test results and evaluation summary for this trained model run.
results_df = pd.DataFrame({
    'window_index': np.arange(len(test_scores)),
    'test_score': test_scores,
    'test_prediction': test_preds,
    'test_label': test_labels,
    'correct': (test_preds == test_labels).astype(int),
})

train_metrics_saved = summary.get('train_metrics', {})
saved_plots = (
    {
        'reconstruction_error_distribution': to_relative_path(threshold_plot_path),
        'anomaly_timeline_with_fn_density': to_relative_path(timeline_plot_path),
    }
    if MODEL_TYPE == 'autoencoder'
    else {
        'score_distribution': to_relative_path(threshold_plot_path),
        'score_timeline': to_relative_path(timeline_plot_path),
    }
)

evaluation_payload = {
    'model_run_name': selected_run_name,
    'model_run_path': to_relative_path(selected_run_path),
    'model_type': MODEL_TYPE,
    'evaluated_at_utc': str(pd.Timestamp.utcnow()),
    'data_source': {
        'source_data_run': to_relative_path(window_run_dir),
    },
    'threshold': float(threshold),
    'training_metrics': train_metrics_saved,
    'test_metrics': metrics,
    'saved_plots': saved_plots,
    'counts': {
        'n_test': int(len(test_labels)),
        'n_actual_anomaly': int((test_labels == 1).sum()),
        'n_predicted_anomaly': int((test_preds == 1).sum()),
        'n_false_negative': int(((test_labels == 1) & (test_preds == 0)).sum()),
    },
}

evaluation_json_path = selected_run_path / 'evaluation.json'
evaluation_json_path.write_text(json.dumps(evaluation_payload, indent=2), encoding='utf-8')

print('Showing first 10 predictions:')
display(results_df.head(10))

print(f'\nAccuracy on test set: {(test_preds == test_labels).mean():.4f}')
print(f'Evaluation summary saved to: {evaluation_json_path}')